<a href="https://colab.research.google.com/github/abrar2akib/Sample-Codes/blob/main/Assignment%20on%20Solar%20Panel/%20Panel_solar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Necessary Installs

In [ ]:
!pip install pyreadstat

# Dataset loading

In [ ]:
import pandas as pd
import pyreadstat

df_V, meta_V = pyreadstat.read_dta('/content/drive/MyDrive/Colab Notebooks/MSS_Thesis/BIHS_all/BIHS_2018/BIHSRound3/Male/073_bihs_r3_male_mod_v2.dta')
df_Q, meta_Q = pyreadstat.read_dta('/content/drive/MyDrive/Colab Notebooks/MSS_Thesis/BIHS_all/BIHS_2018/BIHSRound3/Male/063_bihs_r3_male_mod_q.dta')
df_A, meta_A = pyreadstat.read_dta('/content/drive/MyDrive/Colab Notebooks/MSS_Thesis/BIHS_all/BIHS_2018/BIHSRound3/Male/009_bihs_r3_male_mod_a.dta')
df_B, meta_B = pyreadstat.read_dta('/content/drive/MyDrive/Colab Notebooks/MSS_Thesis/BIHS_all/BIHS_2018/BIHSRound3/Male/010_bihs_r3_male_mod_b1.dta')
df_G, meta_G = pyreadstat.read_dta('/content/drive/MyDrive/Colab Notebooks/MSS_Thesis/BIHS_all/BIHS_2018/BIHSRound3/Male/020_bihs_r3_male_mod_g.dta')
df_D, meta_D = pyreadstat.read_dta('/content/drive/MyDrive/Colab Notebooks/MSS_Thesis/BIHS_all/BIHS_2018/BIHSRound3/Male/015_bihs_r3_male_mod_d1.dta')
df_O, meta_O = pyreadstat.read_dta('/content/drive/MyDrive/Colab Notebooks/MSS_Thesis/BIHS_all/BIHS_2018/BIHSRound3/Female/096_bihs_r3_female_mod_o1.dta')
df_P, meta_P = pyreadstat.read_dta('/content/drive/MyDrive/Colab Notebooks/MSS_Thesis/BIHS_all/BIHS_2018/BIHSRound3/Male/061_bihs_r3_male_mod_p1.dta')

df_V = df_V[['a01','v2_01','v2_06']]
df_Q = df_Q[['a01','q_13','q_14', 'q_15','q_16','q_17','q_18','q_19']]
df_A = df_A[['a01','a23','div']]
df_B = df_B[['a01','b1_01','b1_02','b1_08','b1_11','b1_13a']]
df_G = df_G[['a01']]
df_D = df_D[['a01','d1_02','d1_03']]  # If any solar owned
df_P = df_P[['a01','p1_01','p1_02']]  # Cash expenditure on electricity (7 grid, 46 generator 47 solar)
df_O = df_O[['a01', 'o1_06','o1_07']]

# Renaming variables

In [ ]:
# Example renaming dictionaries
rename_V = {'a01':'hhid','v2_01': 'remittance', 'v2_06': 'remittance_value'}
rename_Q = {'a01':'hhid','q_13': 'electricity_connection', 'q_14': 'load_shedding', 'q_15': 'c_electricity',
            'q_16': 'main_cook_fuel', 'q_17': 'c_cook_fuel', 'q_18': 'main_light_fuel', 'q_19': 'c_light_fuel'}
rename_A = {'a01':'hhid','a23': 'hh_size', 'div': 'division'}
rename_B = {'a01':'hhid','b1_01': 'hhh_gender', 'b1_02': 'hhh_age', 'b1_08': 'hhh_edu',
            'b1_11': 'work_area', 'b1_13a': 'main_earning'}
rename_G = {'a01':'hhid'}
rename_D = {'a01':'hhid','d1_02': 'solar_have', 'd1_03': 'solar_own'}
rename_P = {'a01':'hhid','p1_01': 'exp_on', 'p1_02': 'cash_exp_on'}
rename_O = {'a01':'hhid','o1_06': 'q_food','o1_07':'unit_price'}

# Apply renaming
df_V.rename(columns=rename_V, inplace=True)
df_Q.rename(columns=rename_Q, inplace=True)
df_A.rename(columns=rename_A, inplace=True)
df_B.rename(columns=rename_B, inplace=True)
df_D.rename(columns=rename_D, inplace=True)
df_P.rename(columns=rename_P, inplace=True)
df_O.rename(columns=rename_O, inplace=True)
df_G.rename(columns=rename_G, inplace=True)

# Clean dataframes

## Clean df_V

In [ ]:


# Step 1: Copy and extract base hhid
df_clean_V = df_V.copy()
df_clean_V['base_hhid'] = df_clean_V['hhid'].apply(lambda x: int(float(x)))

# Step 2: Group by base_hhid and apply remittance rule
fin_V = df_clean_V.groupby('base_hhid').agg({
    'remittance': lambda x: 1 if (x == 1).any() else 2
}).reset_index()

# Step 3: Rename column back
fin_V.rename(columns={'base_hhid': 'hhid'}, inplace=True)

# View result
fin_V


,hhid,remittance
0,1,1
1,2,2
2,3,2
3,4,2
4,5,2
...,...,...
5113,5499,2
5114,5500,1
5115,6501,2
5116,6502,2


## Clean df_A

In [ ]:
df_A = df_A.sort_values(by='hhid').reset_index(drop=True)

In [ ]:
# Step 1: Convert hhid to base household ID
df_A['hhid_clean'] = df_A['hhid'].astype(float).astype(int)

# Step 2: Group by base hhid and aggregate
df_A_grouped = df_A.groupby('hhid_clean', as_index=False).agg({
    'hh_size': 'sum',
    'division': 'first'
})

# Step 3: Rename hhid_clean back to hhid (optional)
df_A_grouped = df_A_grouped.rename(columns={'hhid_clean': 'hhid'})

# Step 4: Sort by hhid
df_A_grouped = df_A_grouped.sort_values(by='hhid').reset_index(drop=True)

# View result
df_A_grouped


,hhid,hh_size,division
0,1,3,40
1,2,4,40
2,3,5,40
3,4,5,40
4,5,4,40
...,...,...,...
5498,5499,2,55
5499,5500,3,55
5500,6501,6,55
5501,6502,4,55


## Clean df_Q

In [ ]:
# Step 1: Create base hhid
df_Q['hhid_base'] = df_Q['hhid'].astype(float).astype(int)

# Step 2: Group and aggregate
df_Q_clean = df_Q.groupby('hhid_base', as_index=False).agg({
    'electricity_connection': 'max',  # 1 if any subgroup has 1
    'c_electricity': 'sum',
    'c_cook_fuel': 'sum',
    'c_light_fuel': 'sum'
})
df_Q_clean['total_energy_cost'] = (
    df_Q_clean['c_electricity'].fillna(0) +
    df_Q_clean['c_cook_fuel'].fillna(0) +
    df_Q_clean['c_light_fuel'].fillna(0)
)

# Optional: Rename hhid_base back to hhid
df_Q_clean = df_Q_clean.rename(columns={'hhid_base': 'hhid'})

/tmp/ipython-input-3667704472.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_Q_clean['c_light_fuel'].fillna(0)


## Clean df_B

In [ ]:
# Step 1: Create base hhid (remove .1, .2, etc.)
df_B['hhid_clean'] = df_B['hhid'].astype(float).astype(int)

# Step 2: Keep only the first row for each household
df_B_first = df_B.groupby('hhid_clean', as_index=False).first()

# Step 3: Drop the old hhid column and rename
df_B_first = df_B_first.drop(columns=['hhid']).rename(columns={'hhid_clean': 'hhid'})

# Step 4: Sort if needed
df_B_first = df_B_first.sort_values(by='hhid').reset_index(drop=True)

# View result
df_B_first

,hhid,hhh_gender,hhh_age,hhh_edu,work_area,main_earning
0,1,1,60,99,1,10
1,2,1,54,99,1,5
2,3,1,52,8,1,12
3,4,1,37,99,1,5
4,5,1,60,14,1,5
...,...,...,...,...,...,...
5113,5499,1,68,2,1,8
5114,5500,2,46,2,1,4
5115,6501,1,48,99,1,12
5116,6502,1,34,9,1,4


## Clean df_D

In [ ]:
# Step 1: Create clean hhid
df_D['hhid_clean'] = df_D['hhid'].astype(float).astype(int)

# Step 2: Check condition for own_solar = 1 per group
def check_own_solar(group):
    if ((group['solar_have'] == 43) & (group['solar_own'] == 1)).any():
        return 1
    else:
        return 0

# Step 3: Apply groupby and create new df
df_D_clean = df_D.groupby('hhid_clean').apply(check_own_solar).reset_index()
df_D_clean.columns = ['hhid', 'own_solar']


/tmp/ipython-input-2457935188.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_D_clean = df_D.groupby('hhid_clean').apply(check_own_solar).reset_index()


In [ ]:
# df_D_clean['own_solar'].value_counts()

## CLean df_P

In [ ]:
# Define the list of codes that qualify as energy expenditure
energy_codes = [1, 2, 3, 4, 5, 6, 7, 8, 9, 46, 47]

# Step 1: Create new column `energy_exp` (copy cash_exp_on if exp_on is in list, else 0)
df_P['energy_exp'] = df_P.apply(
    lambda row: row['cash_exp_on'] if row['exp_on'] in energy_codes else 0, axis=1)

# Step 2: Group by hhid and sum both expenditures
df_P_grouped = df_P.groupby('hhid', as_index=False)[['cash_exp_on', 'energy_exp']].sum()

# Step 3: Rename cash_exp_on to nonfood_exp
df_P_grouped.rename(columns={'cash_exp_on': 'nonfood_exp'}, inplace=True)

In [ ]:


# # Sample df_P (replace this with your actual DataFrame)
# # df_P = pd.read_csv(...) or already defined

# # Step 1: Clean hhid
# df_P['hhid_base'] = df_P['hhid'].astype(float).astype(int)

# # Step 2: Create binary columns
# df_P['elec_grid'] = (df_P['exp_on'] == 7).astype(int)
# df_P['elec_gen'] = (df_P['exp_on'] == 46).astype(int)
# df_P['elec_solar'] = (df_P['exp_on'] == 47).astype(int)

# # Step 3: Conditional cash expenditure columns
# df_P['cash_exp_grid'] = df_P['cash_exp_on'].where(df_P['exp_on'] == 7)
# df_P['cash_exp_gen'] = df_P['cash_exp_on'].where(df_P['exp_on'] == 46)
# df_P['cash_exp_solar'] = df_P['cash_exp_on'].where(df_P['exp_on'] == 47)

# # Step 4: Group by base hhid
# df_P_clean = df_P.groupby('hhid_base', as_index=False).agg({
#     'elec_grid': 'max',
#     'elec_gen': 'max',
#     'elec_solar': 'max',
#     'cash_exp_grid': 'sum',
#     'cash_exp_gen': 'sum',
#     'cash_exp_solar': 'sum'
# })

# # Optional: Rename hhid_base back to hhid
# df_P_clean = df_P_clean.rename(columns={'hhid_base': 'hhid'})

# Result: df_P_clean contains one row per household with flags and expenditure


## Clean df_O

In [ ]:
# Step 1: Create baseid using provided method
df_O['hhid_base'] = df_O['hhid'].astype(float).astype(int)

# Step 2: Multiply the two expense columns to get food_exp
df_O['food_exp'] = df_O['q_food'] * df_O['unit_price']

# Step 3: Group by baseid and aggregate by sum
df_O_grouped = df_O.groupby('hhid_base', as_index=False)['food_exp'].sum()
# Optional: Rename hhid_base back to hhid
df_O_grouped = df_O_grouped.rename(columns={'hhid_base': 'hhid'})

# Merging

In [ ]:
del master_df

In [ ]:
# Start with the base DataFrame
master_df = fin_V.copy()

# List of all other DataFrames to merge
dfs_to_merge = [df_A_grouped, df_Q_clean, df_B_first, df_D_clean, df_P_grouped, df_O_grouped]

# Merge each DataFrame on 'hhid' using left join to keep base (fin_V) households
for df in dfs_to_merge:
    master_df = master_df.merge(df, on='hhid', how='left')


/tmp/ipython-input-2940510202.py:9: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  master_df = master_df.merge(df, on='hhid', how='left')


### land owners

In [ ]:
# # Step 1: Clean column names in case of hidden spaces
# df_G.columns = df_G.columns.str.strip()
# master_df.columns = master_df.columns.str.strip()

# # Step 2: Create hhid_base in df_G by converting float-like hhid to int
# df_G['hhid_base'] = df_G['hhid'].astype(float).astype(int)

# # Step 3: Get list of unique base hhids from df_G
# land_owner_ids = df_G['hhid_base'].unique()

# # Step 4: Create a 'land_own' column in master_df
# #         If hhid in master_df is in the land_owner_ids, set to 1, else 0
# master_df['land_own'] = master_df['hhid'].astype(int).isin(land_owner_ids).astype(int)

### Energy poverty

In [ ]:
# First calculate total expenditure
master_df['total_exp'] = master_df['food_exp'] + master_df['nonfood_exp']

# Then identify energy poor households
master_df['energy_poor'] = (master_df['energy_exp'] > 0.05* master_df['total_exp']).astype(int)

In [ ]:
master_df['burden'] = (master_df['c_electricity'] > 0.5 * master_df['energy_exp']).astype(int)

In [ ]:
master_df['burden'].value_counts()

,count
burden,
1,2620
0,2498


In [ ]:
# master_df.drop(columns=['energy_poor'], inplace=True)

In [ ]:
master_df['energy_poor'].value_counts()

,count
energy_poor,
0,5107
1,11


In [ ]:
list(master_df.columns)

['hhid',
 'remittance',
 'hh_size',
 'division',
 'electricity_connection',
 'c_electricity',
 'c_cook_fuel',
 'c_light_fuel',
 'total_energy_cost',
 'hhh_gender',
 'hhh_age',
 'hhh_edu',
 'work_area',
 'main_earning',
 'own_solar',
 'nonfood_exp',
 'energy_exp',
 'food_exp',
 'land_own',
 'total_exp',
 'energy_poor']

In [ ]:
master_df['c_electricity'].value_counts(30)

,proportion
c_electricity,
0.0,0.162954
200.0,0.038296
150.0,0.030481
300.0,0.030090
100.0,0.028918
...,...
668.0,0.000195
347.0,0.000195
273.0,0.000195


In [ ]:
export_excel = master_df.to_excel('master_df.xlsx')

#Solar Own

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
import numpy as np

# Assuming master_df is your dataframe with columns like:
# ['hhid', 'own_solar', 'burden', 'division', 'hhh_edu', 'main_earning', ... ]

# --- Step 1: Prepare variables ---

treatment = 'own_solar'     # Binary treatment variable (1 = owns solar, 0 = does not)
outcome = 'burden'          # Outcome variable (e.g., energy burden or similar)
controls = ['hh_size', 'division', 'hhh_edu', 'work_area', 'main_earning', 'hhh_gender']  # Control variables

# Replace 2 with 0 in binary vars if needed (assuming 1 = yes, 2 = no)
for var in ['own_solar', 'electricity_connection', 'work_area', 'hhh_gender']:
    if var in master_df.columns:
        master_df[var] = master_df[var].replace({2: 0})

# --- Step 2: Subset and clean data ---
psm_df = master_df[[treatment, outcome] + controls].dropna().copy()

# --- Step 3: One-hot encode categorical controls ---
categorical_vars = ['division', 'hhh_edu', 'main_earning']
psm_df = pd.get_dummies(psm_df, columns=categorical_vars, drop_first=True)

# Remove duplicate columns if any
psm_df = psm_df.loc[:, ~psm_df.columns.duplicated()]

# --- Step 4: Define X and y for propensity score model ---
X = psm_df.drop(columns=[treatment, outcome])
y = psm_df[treatment]

# --- Step 5: Fit logistic regression to get propensity scores ---
model = LogisticRegression(max_iter=1000)
model.fit(X, y)
pscores = model.predict_proba(X)[:, 1]

# --- Step 6: Perform Nearest Neighbor matching ---
treated_idx = psm_df[psm_df[treatment] == 1].index
control_idx = psm_df[psm_df[treatment] == 0].index

treated_pos = psm_df.index.get_indexer(treated_idx)
control_pos = psm_df.index.get_indexer(control_idx)

nn = NearestNeighbors(n_neighbors=1)
nn.fit(pscores[control_pos].reshape(-1, 1))

distances, indices = nn.kneighbors(pscores[treated_pos].reshape(-1, 1))

matched_control_pos = control_pos[indices.flatten()]
matched_treated_pos = treated_pos

matched_control_idx = psm_df.index[matched_control_pos]
matched_treated_idx = psm_df.index[matched_treated_pos]

att = (psm_df.loc[matched_treated_idx, outcome].values - psm_df.loc[matched_control_idx, outcome].values).mean()
print(f"Estimated ATT from PSM: {att:.4f}")


Estimated ATT from PSM: -0.2744


In [ ]:
import numpy as np

# Number of bootstrap samples
n_boot = 1000

att_bootstrap = []

for _ in range(n_boot):
    # Sample with replacement from matched treated indices
    sample_idx = np.random.choice(len(matched_treated_idx), size=len(matched_treated_idx), replace=True)

    treated_sample = psm_df.loc[matched_treated_idx].iloc[sample_idx]
    control_sample = psm_df.loc[matched_control_idx].iloc[sample_idx]

    att_sample = (treated_sample[outcome].values - control_sample[outcome].values).mean()
    att_bootstrap.append(att_sample)

# Calculate 95% confidence interval
lower = np.percentile(att_bootstrap, 2.5)
upper = np.percentile(att_bootstrap, 97.5)

print(f"ATT: {att:.4f}")
print(f"95% Confidence Interval: [{lower:.4f}, {upper:.4f}]")

if lower > 0 or upper < 0:
    print("The ATT is statistically significant at 5% level.")
else:
    print("The ATT is NOT statistically significant at 5% level.")


ATT: -0.2744
95% Confidence Interval: [-0.3201, -0.2274]
The ATT is statistically significant at 5% level.


#Propensity Score Matching

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
import numpy as np

# Assuming master_df is your dataframe with columns like:
# ['hhid', 'remittance', 'energy_poor', 'division', 'hhh_edu', 'main_earning', ... ]

# --- Step 1: Prepare variables ---

treatment = 'remittance'     # Binary treatment variable
outcome = 'energy_poor'      # Outcome variable
controls = ['hh_size', 'division', 'hhh_edu', 'work_area', 'main_earning', 'hhh_gender']  # Add your control variables here

# Replace 2 with 0 in binary vars if needed (assuming 1 = yes, 2 = no)
for var in ['remittance', 'electricity_connection', 'work_area', 'hhh_gender']:
    if var in master_df.columns:
        master_df[var] = master_df[var].replace({2: 0})

# --- Step 2: Subset and clean data ---
psm_df = master_df[[treatment, outcome] + controls].dropna().copy()

# --- Step 3: One-hot encode categorical controls ---
categorical_vars = ['division', 'hhh_edu', 'main_earning']
psm_df = pd.get_dummies(psm_df, columns=categorical_vars, drop_first=True)

# Remove duplicate columns if any (fixes y 2D issue)
psm_df = psm_df.loc[:, ~psm_df.columns.duplicated()]

# --- Step 4: Define X and y for propensity score model ---
X = psm_df.drop(columns=[treatment, outcome])
y = psm_df[treatment]

# --- Step 5: Fit logistic regression to get propensity scores ---
model = LogisticRegression(max_iter=1000)
model.fit(X, y)
pscores = model.predict_proba(X)[:, 1]

# --- Step 6: Perform Nearest Neighbor matching ---
treated_idx = psm_df[psm_df[treatment] == 1].index
control_idx = psm_df[psm_df[treatment] == 0].index

# Map index labels to positional integer locations for pscores numpy array
treated_pos = psm_df.index.get_indexer(treated_idx)
control_pos = psm_df.index.get_indexer(control_idx)

nn = NearestNeighbors(n_neighbors=1)
nn.fit(pscores[control_pos].reshape(-1, 1))

distances, indices = nn.kneighbors(pscores[treated_pos].reshape(-1, 1))

matched_control_pos = control_pos[indices.flatten()]
matched_treated_pos = treated_pos

# Now get back to index labels for dataframe access
matched_control_idx = psm_df.index[matched_control_pos]
matched_treated_idx = psm_df.index[matched_treated_pos]

att = (psm_df.loc[matched_treated_idx, outcome].values - psm_df.loc[matched_control_idx, outcome].values).mean()
print(f"Estimated ATT from PSM: {att:.4f}")


Estimated ATT from PSM: 0.0016


In [ ]:
import numpy as np

# Assuming you have these from your PSM step:
# matched_treated_idx, matched_control_idx, psm_df, outcome

treated_outcomes = psm_df.loc[matched_treated_idx, outcome].values
control_outcomes = psm_df.loc[matched_control_idx, outcome].values

n_boot = 1000
boot_atts = []

np.random.seed(42)

for _ in range(n_boot):
    sample_idx = np.random.choice(len(treated_outcomes), len(treated_outcomes), replace=True)
    boot_att = np.mean(treated_outcomes[sample_idx] - control_outcomes[sample_idx])
    boot_atts.append(boot_att)

# 90% CI: 5th and 95th percentiles
ci_lower = np.percentile(boot_atts, 5)
ci_upper = np.percentile(boot_atts, 95)

print(f"Bootstrap 90% CI for ATT: [{ci_lower:.4f}, {ci_upper:.4f}]")

if ci_lower > 0 or ci_upper < 0:
    print("ATT is statistically significant at 10% level.")
else:
    print("ATT is NOT statistically significant at 10% level.")


Bootstrap 90% CI for ATT: [0.0000, 0.0031]
ATT is NOT statistically significant at 10% level.


In [ ]:
print(psm_df.columns[psm_df.columns.duplicated()])


Index(['remittance'], dtype='object')


In [ ]:
# --- Diagnostics ---
print(f"Treated units: {len(matched_treated_idx)}, Matched control units: {len(matched_control_idx)}")
print("Propensity score range (treated):", round(pscores[treated_idx].min(), 3), "-", round(pscores[treated_idx].max(), 3))
print("Propensity score range (control):", round(pscores[control_idx].min(), 3), "-", round(pscores[control_idx].max(), 3))

NameError: name 'matched_treated_idx' is not defined

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# STEP 1: Load your data and define the relevant variables
# Assuming your DataFrame is called master_df

# Define treatment, outcome, and control variables
treatment = 'remittance'
outcome = 'energy_poor'
controls = [
    'hh_size',
    'division',  # categorical
    'electricity_connection',
    'c_electricity',
    'c_cook_fuel',
    'c_light_fuel',
    'hhh_gender',
    'hhh_age',
    'hhh_edu',
    'work_area',  # categorical
    'main_earning',
    'own_solar',
    'food_exp',
    'nonfood_exp'
]

# STEP 2: Subset and preprocess the data
psm_df = master_df[[treatment, outcome] + controls].dropna().copy()

# One-hot encode categorical variables
psm_df = pd.get_dummies(psm_df, columns=['division', 'work_area'], drop_first=True)

# STEP 3: Define X (covariates), T (treatment), and Y (outcome)
X = psm_df.drop(columns=[treatment, outcome])
T = psm_df[treatment]
Y = psm_df[outcome]

# Optional: Standardize covariates (improves matching performance)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# STEP 4: Estimate Propensity Scores using Logistic Regression
logit = LogisticRegression(max_iter=1000)
logit.fit(X_scaled, T)
pscores = logit.predict_proba(X_scaled)[:, 1]  # Propensity scores

# STEP 5: Perform Nearest Neighbor Matching (1:1 without replacement)

# Get indices for treated and control
treated_idx = T[T == 1].index
control_idx = T[T == 0].index

# Fit Nearest Neighbors on control group propensity scores
nn = NearestNeighbors(n_neighbors=1)
nn.fit(pscores[control_idx].reshape(-1, 1))

# Find nearest neighbors for each treated unit
distances, indices = nn.kneighbors(pscores[treated_idx].reshape(-1, 1))
matched_control_idx = control_idx[indices.flatten()]
matched_treated_idx = treated_idx

# STEP 6: Estimate ATE
treated_outcomes = Y.loc[matched_treated_idx].values
control_outcomes = Y.loc[matched_control_idx].values
ATE = np.mean(treated_outcomes - control_outcomes)

print(f"Estimated ATE (effect of receiving remittance on being energy poor): {ATE:.4f}")


KeyError: "['division', 'hhh_edu', 'main_earning'] not in index"

# Inspection

##Inspecting variable name and labels

In [ ]:
# Show full question labels in output
pd.set_option('display.max_colwidth', None)

def create_label_table_by_letter(letter):
    df = eval(f'df_{letter}')
    meta = eval(f'meta_{letter}')

    # If column_labels is a list
    try:
        labels = [meta.column_labels[df.columns.get_loc(col)] for col in df.columns]
    except (AttributeError, IndexError, KeyError, TypeError):
        labels = ['Label not found' for _ in df.columns]

    return pd.DataFrame({
        'Variable Name': df.columns,
        'Label (Question Text)': labels
    })

In [ ]:
create_label_table_by_letter('A')

,Variable Name,Label (Question Text)
0,hhid,Household identification number (numeric)
1,hh_size,STime Start time (hour) of Module A & Consent
2,division,STime Start time (minute) of Module A & Consent
3,hhid_clean,Respondent ID for Module A


## To inspect the hh without land

In [ ]:
# Step 1: Extract unique, sorted values
a01_values = sorted(df_G['a01'].dropna().unique())

# Step 2: Generate full expected range
full_range = set(range(int(min(a01_values)), int(max(a01_values)) + 1))

# Step 3: Identify missing values
present = set(int(x) for x in a01_values)
missing = sorted(full_range - present)

# Step 4: Output summary
print(f"Min a01: {min(a01_values)}")
print(f"Max a01: {max(a01_values)}")
print(f"Missing values: {missing}")
print(f"Count of missing values: {len(missing)}")


KeyError: 'a01'

# Merging modules

In [ ]:
df_V.head(50)

# Test Lab


In [ ]:


# Extended test data with .1, .2, .3 versions
data = {
    'hhid': [
        1.0,                               # Only one entry → 1
        2.1, 2.2, 2.3,                     # All remittance = 2 → keep 2
        3.1, 3.2, 3.3,                     # All remittance = 1 → keep 1
        4.1, 4.2, 4.3,                     # Mixed (1, 2, 2) → keep 1
        5.1, 5.2,                          # Mixed (1, 2) → keep 1
        6.1, 6.2,                          # All 2 → keep 2
        7.1, 7.2,                          # All 1 → keep 1
        8.0                                # Only one entry → 2
    ],
    'remittance': [
        1,
        2, 2, 2,
        1, 1, 1,
        1, 2, 2,
        1, 2,
        2, 2,
        1, 1,
        2
    ],
    'remittance_value': [
        100000,
        None, None, None,
        300000, 150000, 200000,
        100000, None, None,
        100000, None,
        None, None,
        None, None,
        None
    ]
}


df_tl= pd.DataFrame(data)
df_tl